In [1]:
!pip install datasets
import re, random
import pandas as pd
from datasets import load_dataset

random.seed(42)

# Source 1: short movie review sentences (casual, chat-like text)
# rt = load_dataset("rotten_tomatoes", split="train").to_pandas()
rt = load_dataset("cornell-movie-review-data/rotten_tomatoes", split="train").to_pandas()
rt_rows = [{"text": t.strip(), "label": "safe", "source": "rotten_tomatoes"}
           for t in rt["text"].sample(3000, random_state=42)]

# Source 2: news sentences (formal text)
# news = load_dataset("ag_news", split="train").to_pandas()
news = load_dataset("fancyzhx/ag_news", split="train").to_pandas()
news_rows = []
for t in news["text"].sample(3000, random_state=42):
    for s in re.split(r'(?<=[.!?])\s+', t):
        if 20 <= len(s) <= 120:
            news_rows.append({"text": s.strip(), "label": "safe", "source": "ag_news"})

# Source 3: code and UI lines (so normal code screenshots are not flagged)
code_lines = [
    "import pandas as pd", "def main():", "for i in range(10):", "print('Hello world')",
    "return result", "if __name__ == '__main__':", "x = df.groupby('label').size()",
    "class UserProfile:", "git commit -m 'update readme'", "npm install express",
    "pip install scikit-learn", "SELECT * FROM orders WHERE id = 5;", "Loading, please wait...",
    "File  Edit  View  Help", "Save changes?", "Welcome back!", "Terms and Conditions",
    "Search settings", "Download complete", "Error: file not found", "Unread messages (3)",
    "Python 3.12.4", "Build succeeded in 4.2s", "Page 1 of 10", "Dark mode enabled",
    "Updated 3 minutes ago", "Press Enter to continue", "Total items: 25", "Status: Active",
    "Run all cells", "Install complete", "Copyright 2026 All rights reserved",
]
code_rows = [{"text": c, "label": "safe", "source": "ui_code"} for c in code_lines]

extra_safe = pd.DataFrame(rt_rows + news_rows + code_rows)
print(extra_safe.shape)
extra_safe.sample(6, random_state=1)


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


c:\Users\Victus\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(4436, 3)


,text,label,source
453,"what i saw , i enjoyed .",safe,rotten_tomatoes
364,the most brilliant work in this genre since th...,safe,rotten_tomatoes
372,"like a skillful fisher , the director uses the...",safe,rotten_tomatoes
1637,the plot plummets into a comedy graveyard befo...,safe,rotten_tomatoes
857,"the production design , score and choreography...",safe,rotten_tomatoes
2204,"slack and uninspired , and peopled mainly by c...",safe,rotten_tomatoes


In [2]:
def looks_sensitive(t):
    return bool(re.search(r"@|http|www\.|\d{7,}", t))

before = len(extra_safe)
extra_safe = extra_safe[~extra_safe["text"].apply(looks_sensitive)]
print("removed:", before - len(extra_safe))

removed: 0


In [3]:
syn = pd.read_csv("../data/synthetic.csv")
pub = pd.read_csv("../data/ai4privacy.csv")

df = pd.concat([syn, pub, extra_safe], ignore_index=True)
print("raw merged:", df.shape)

raw merged: (18936, 3)


In [4]:
# 1. missing values
print(df.isna().sum())
df = df.dropna(subset=["text", "label"])

# 2. whitespace and length
df["text"] = df["text"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
df = df[(df["text"].str.len() >= 3) & (df["text"].str.len() <= 200)]

# 3. duplicates
print("duplicates:", df.duplicated(subset="text").sum())
df = df.drop_duplicates(subset="text")

# 4. shuffle and save
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.to_csv("../data/clean_dataset.csv", index=False)

print(df.shape)
print(df["label"].value_counts())
print(pd.crosstab(df["source"], df["label"]))

text      0
label     0
source    0
dtype: int64
duplicates: 1700
(17080, 3)
label
safe            8622
credential      2700
email           2500
phone           2500
url_internal     758
Name: count, dtype: int64
label            credential  email  phone  safe  url_internal
source                                                       
ag_news                   0      0      0  1402             0
ai4privacy             1500   1500   1500     0             0
faker_synthetic        1200   1000   1000  4344           758
rotten_tomatoes           0      0      0  2844             0
ui_code                   0      0      0    32             0


# 03 - Merge and Clean

## Purpose
Combine all sources into one clean dataset and add realistic "safe" text, so safe is not
only Faker sentences.

## Input / Output
- **Input:** `synthetic.csv`, `ai4privacy.csv`, plus safe text loaded from Hugging Face
- **Output:** `data/clean_dataset.csv` (about 17,200 rows)

## What we did
1. Loaded extra safe text:
   - Rotten Tomatoes movie review sentences (casual text)
   - AG News sentences (formal text), split into lines of 20 to 120 characters
   - 32 hand-written UI and code lines (e.g. `import pandas as pd`)
2. Removed "safe" lines that contain `@`, `http`, `www.` or 7+ digit numbers, because
   they would carry wrong labels.
3. Merged the three sources into one table.
4. Cleaned the data:
   - checked missing values (none found)
   - normalized whitespace and kept text length between 3 and 200 characters
   - removed 1,565 duplicates
5. Shuffled and saved the result, then printed the class counts and a source-by-label
   table.

## Final class counts
| Class | Rows |
|---|---|
| safe | 8,756 |
| credential | 2,700 |
| email | 2,500 |
| phone | 2,500 |
| url_internal | about 760 |

## Key concepts learned
- **Merging** datasets needs a common format (text, label, source).
- **Cleaning** steps: missing values, whitespace, length limits, duplicates.
- Wrong labels hurt a model more than missing data, so we filtered suspicious safe lines.
- The **source column** lets us later check whether the model relies on source style.
- We deliberately did not balance the data here. Imbalance is handled in modeling.

## Problems and fixes
- Hugging Face dataset names needed full IDs (`cornell-movie-review-data/rotten_tomatoes`,
  `fancyzhx/ag_news`).

## Brief requirement covered
Preprocessing (missing values, inconsistencies), use of at least two datasets,
data description.